# HackRare — Fine-Tune Llama 3.1 8B for Drug Repurposing

Fine-tunes `meta-llama/Meta-Llama-3.1-8B-Instruct` on the rare-disease drug repurposing
dataset using **QLoRA** (4-bit NF4 + LoRA adapters).

**Run cells top to bottom. Edit Cell 3 (config) before running anything else.**

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive/MyDrive/')

In [ ]:
# ── Cell 2: Install Dependencies ────────────────────────────────────────────
# Do NOT restart runtime after this — just continue to the next cell
%pip install -q \
    transformers==4.45.2 \
    peft==0.13.2 \
    trl==0.11.4 \
    bitsandbytes==0.44.1 \
    accelerate==1.0.1 \
    datasets==3.1.0 \
    scipy \
    huggingface_hub
print('Installation complete.')

In [ ]:
# ── Cell 3: Configuration  ← EDIT THIS CELL ─────────────────────────────────

HF_TOKEN    = "hf_YOUR_TOKEN_HERE"          # Your Hugging Face token (needs Llama access)
DRIVE_DIR   = "/content/drive/MyDrive/hackrare"          # Folder you created in Drive
OUTPUT_DIR  = "/content/drive/MyDrive/hackrare/llama-hackrare-adapter"  # Final adapter save path
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackrare/checkpoints"  # Mid-training checkpoints (resumable)
MODEL_NAME  = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# ── Training hyperparameters ──
MAX_SEQ_LEN   = 1024   # Reduce to 512 if on T4 (free tier)
BATCH_SIZE    = 2      # Reduce to 1 if on T4
GRAD_ACCUM    = 8      # Effective batch = BATCH_SIZE * GRAD_ACCUM = 16
EPOCHS        = 3
LEARNING_RATE = 2e-4
WARMUP_RATIO  = 0.03

# ── LoRA hyperparameters ──
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Resume from checkpoint? ──
# Set to True if a previous Colab session was interrupted and you want to resume
RESUME_FROM_CHECKPOINT = False

import os
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Login to Hugging Face ──
from huggingface_hub import login
login(token=HF_TOKEN)
print(f"Logged in.")
print(f"Data dir       : {DRIVE_DIR}")
print(f"Checkpoints dir: {CHECKPOINTS_DIR}")
print(f"Final output   : {OUTPUT_DIR}")

In [ ]:
# ── Cell 4: Load and Format Dataset ─────────────────────────────────────────
import json
import os
from pathlib import Path
from datasets import Dataset
from tqdm.auto import tqdm

SYSTEM_PROMPT = (
    "You are a rare disease drug repurposing expert. "
    "Given a drug-disease pair, reason step-by-step through mechanism compatibility, "
    "target/pathway overlap, safety, and human evidence, then give a final assessment."
)

def format_example(row: dict) -> dict:
    """Wrap one training example in the Llama 3.1 chat template."""
    cot  = str(row.get("chain_of_thought", ""))
    out  = str(row.get("output", ""))
    assistant_text = cot + "\n\n" + out if cot else out

    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}"
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{str(row.get('input', ''))}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{assistant_text}"
        "<|eot_id|>"
    )
    return {"text": text, "label": int(row.get("label", 0))}

# Load JSONL with tqdm progress
jsonl_path = os.path.join(DRIVE_DIR, "training_examples.jsonl")
assert os.path.exists(jsonl_path), f"Not found: {jsonl_path} — check Drive folder name"

raw_data = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in tqdm(lines, desc="Loading JSONL", unit="examples"):
    line = line.strip()
    if line:
        raw_data.append(json.loads(line))

print(f"Loaded {len(raw_data)} raw examples. Formatting ...")
formatted = [format_example(r) for r in tqdm(raw_data, desc="Formatting", unit="examples")]
dataset   = Dataset.from_list(formatted)

# Train / val split (90/10)
split         = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"\nTrain examples : {len(train_dataset)}")
print(f"Eval  examples : {len(eval_dataset)}")
print("\nSample (truncated):")
print(train_dataset[0]["text"][:400])

In [ ]:
# ── Cell 5: Load Base Model with 4-bit QLoRA ─────────────────────────────────
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer from {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading model in 4-bit from {MODEL_NAME} (this takes ~5-10 min on first run) ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache           = False
model.config.pretraining_tp      = 1
print("Model loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── Cell 6: Apply LoRA Adapters ───────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Cell 7: Train ─────────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, TrainerState, TrainerControl
from tqdm.auto import tqdm
import math

# ── Custom tqdm progress callback ──────────────────────────────────────────
class TqdmTrainingCallback(TrainerCallback):
    """Shows two tqdm bars: one per-epoch, one per-step within each epoch."""

    def __init__(self):
        self.epoch_bar  = None
        self.step_bar   = None
        self._steps_per_epoch = None

    def on_train_begin(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        total_steps       = state.max_steps
        self._steps_per_epoch = math.ceil(total_steps / args.num_train_epochs)
        self.epoch_bar = tqdm(
            total=int(args.num_train_epochs),
            desc="Epochs",
            position=0,
            leave=True,
        )
        self.step_bar = tqdm(
            total=self._steps_per_epoch,
            desc="Epoch 1 steps",
            position=1,
            leave=False,
        )

    def on_step_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        self.step_bar.update(1)
        if state.log_history:
            last_loss = next(
                (h["loss"] for h in reversed(state.log_history) if "loss" in h), None
            )
            if last_loss is not None:
                self.step_bar.set_postfix(loss=f"{last_loss:.4f}")

    def on_epoch_begin(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        epoch_num = int(state.epoch) + 1 if state.epoch is not None else 1
        self.step_bar.reset()
        self.step_bar.set_description(f"Epoch {epoch_num} steps")

    def on_epoch_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        self.epoch_bar.update(1)
        train_loss = next(
            (h["loss"] for h in reversed(state.log_history) if "loss" in h), None
        )
        eval_loss = next(
            (h["eval_loss"] for h in reversed(state.log_history) if "eval_loss" in h), None
        )
        postfix = {}
        if train_loss is not None:
            postfix["train_loss"] = f"{train_loss:.4f}"
        if eval_loss is not None:
            postfix["eval_loss"] = f"{eval_loss:.4f}"
        if postfix:
            self.epoch_bar.set_postfix(**postfix)

    def on_train_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        if self.step_bar:
            self.step_bar.close()
        if self.epoch_bar:
            self.epoch_bar.close()
        print("\nTraining complete.")
        # Print final loss summary
        if state.log_history:
            train_loss = next(
                (h["loss"] for h in reversed(state.log_history) if "loss" in h), None
            )
            eval_loss = next(
                (h["eval_loss"] for h in reversed(state.log_history) if "eval_loss" in h), None
            )
            if train_loss: print(f"  Final train loss : {train_loss:.4f}")
            if eval_loss:  print(f"  Final eval  loss : {eval_loss:.4f}")

# ── Training config ─────────────────────────────────────────────────────────
training_args = SFTConfig(
    output_dir=CHECKPOINTS_DIR,          # Mid-training checkpoints go here (resumable)
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,              # Keep 3 most recent checkpoints on Drive
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    disable_tqdm=True,               # Disable default HF tqdm — our callback handles it
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
    callbacks=[TqdmTrainingCallback()],
)

# ── Resume from checkpoint if requested ─────────────────────────────────────
resume = None
if RESUME_FROM_CHECKPOINT:
    checkpoints = sorted(
        [d for d in os.listdir(CHECKPOINTS_DIR) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[-1])
    )
    if checkpoints:
        resume = os.path.join(CHECKPOINTS_DIR, checkpoints[-1])
        print(f"Resuming from checkpoint: {resume}")
    else:
        print("No checkpoints found — starting from scratch.")

print(f"Training on {len(train_dataset)} examples for {EPOCHS} epochs ...")
print(f"Checkpoints saving to: {CHECKPOINTS_DIR}")
trainer.train(resume_from_checkpoint=resume)

In [ ]:
# ── Cell 8: Save Adapter to Drive ────────────────────────────────────────────
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Also save training metadata
import json
meta = {
    "base_model": MODEL_NAME,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "epochs": EPOCHS,
    "max_seq_len": MAX_SEQ_LEN,
    "train_examples": len(train_dataset),
    "eval_examples": len(eval_dataset),
}
with open(os.path.join(OUTPUT_DIR, "training_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"Adapter saved to: {OUTPUT_DIR}")
print("Files saved:")
for fname in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {fname}  ({size_mb:.1f} MB)")

In [ ]:
# ── Cell 9: Test Inference ────────────────────────────────────────────────────
import torch
from peft import PeftModel

# Reload adapter on top of the quantized base (already in memory from training)
model.eval()

def ask_model(drug: str, disease: str, max_new_tokens: int = 400) -> str:
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}"
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"Evaluate drug repurposing candidate:\nDrug: {drug}\nDisease: {disease}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # Return only the assistant's reply
    if "assistant" in decoded:
        return decoded.split("assistant")[-1].strip()
    return decoded

# ── Test examples ──
test_cases = [
    ("Imatinib",       "Duchenne muscular dystrophy"),
    ("Sildenafil",     "pulmonary arterial hypertension"),
    ("Ivacaftor",      "cystic fibrosis"),
    ("Betamethasone",  "hereditary angioedema"),
]

for drug, disease in test_cases:
    print(f"\n{'='*60}")
    print(f"Drug: {drug}  |  Disease: {disease}")
    print('='*60)
    response = ask_model(drug, disease)
    print(response)

In [ ]:
# ── Cell 10 (Optional): Reload Adapter from Drive in a Fresh Session ──────────
# Run this cell if you want to load your saved adapter without re-training

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

RELOAD_BASE   = "meta-llama/Meta-Llama-3.1-8B-Instruct"
RELOAD_ADAPTER = "/content/drive/MyDrive/hackrare/llama-hackrare-adapter"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(RELOAD_ADAPTER)
base_model = AutoModelForCausalLM.from_pretrained(
    RELOAD_BASE,
    quantization_config=bnb_cfg,
    device_map="auto",
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
)
fine_tuned = PeftModel.from_pretrained(base_model, RELOAD_ADAPTER)
fine_tuned.eval()
print("Adapter reloaded successfully.")